In [0]:
%skip
from datetime import datetime
from pyspark.sql import functions as F
dt = datetime.now().strftime("%Y%m%d%H%M%S")
schema_location = "/Volumes/facttransaction/bronze/checkpoint/schema/experiment_" + dt
checkpoint_location = "/Volumes/facttransaction/bronze/checkpoint/stream/experiment_" + dt
dbutils.fs.mkdirs(schema_location)
dbutils.fs.mkdirs(checkpoint_location)

df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("header", "true")
        .load("/Volumes/facttransaction/bronze/20260922/")
)

def process_batch(batch_df, batch_id):

    file_summary = (
        batch_df
        .groupBy("_metadata.file_path")
        .agg(
            F.count("*").alias("row_count")
        )
        .select(
            F.lit(batch_id).alias("batch_id"),
            F.element_at(
                F.split(F.col("file_path"), "/"),
                -1
            ).alias("file_name"),
            F.col("row_count")
        )
    )

    file_summary.write \
        .mode("overwrite") \
        .saveAsTable(
            "facttransaction.bronze.foreach_test_log"
        )

query = (
    df.writeStream
      .foreachBatch(process_batch)
      .option("checkpointLocation", checkpoint_location)
      .trigger(availableNow=True)
      .start()
)

query.awaitTermination()


In [0]:
from datetime import datetime
from pyspark.sql import functions as F

dt = datetime.now().strftime("%Y%m%d%H%M%S")
schema_location = "/Volumes/facttransaction/bronze/checkpoint/schema/dynamic_" + dt
checkpoint_location = "/Volumes/facttransaction/bronze/checkpoint/stream/dynamic_" + dt
dbutils.fs.mkdirs(schema_location)
dbutils.fs.mkdirs(checkpoint_location)

df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("header", "true")
        .load("/Volumes/facttransaction/bronze/20260922/")
)

def process_batch(batch_df, batch_id):
    # Get distinct file paths in this batch
    files = (
        batch_df
        .select(F.col("_metadata.file_path").alias("fp"))
        .distinct()
        .collect()
    )

    for row in files:
        file_path = row.fp
        file_name = file_path.rsplit("/", 1)[-1]
        table_name = "facttransaction.bronze." + file_name.rsplit(".", 1)[0]

        print(f"Writing {file_name} -> {table_name}")

        # Create table if not exists, append if exists
        table_exists = spark.catalog.tableExists(table_name)
        write_mode = "append" if table_exists else "overwrite"

        (
            batch_df
            .filter(F.col("_metadata.file_path") == file_path)
            .drop("_metadata")
            .withColumn("insert_date", F.current_timestamp())
            .write
            .mode(write_mode)
            .saveAsTable(table_name)
        )

query = (
    df.writeStream
      .foreachBatch(process_batch)
      .option("checkpointLocation", checkpoint_location)
      .trigger(availableNow=True)
      .start()
)

query.awaitTermination()
print("Stream completed")

In [0]:
%skip
dbutils.widgets.text("file_location", "")
file_location = dbutils.widgets.get("file_location")
print(file_location)
parts = file_location.rstrip("/").split("/")
print(parts)
table_name = parts[-2]
load_date = parts[-1]

file_name = parts[-2] + "_" + parts[-1]
file_path = file_location + file_name + ".csv"
print(file_path)
table_path = "facttransaction.bronze." + table_name
print(table_path)


# dbutils.widgets.text("table_name", "")
# file_path = "/Volumes/facttransaction/bronze/20260922/" + dbutils.widgets.get("table_name") + ".csv"
# print(file_path)
# table_path = "facttransaction.bronze." + dbutils.widgets.get("table_name")

# spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_path).distinct().write.mode("append").saveAsTable(table_path)